In [1]:
import pandas as pd
from gensim.models import Word2Vec

# Load dataset
train_df = pd.read_csv("train_hate.csv")

# Tokenize sentences (simple split method)
train_tokens = [sentence.lower().split() for sentence in train_df["Sentence"]]

# Train Word2Vec model
embedding_size = 100
word2vec_model = Word2Vec(sentences=train_tokens, vector_size=embedding_size, window=5, min_count=1, workers=4)

# Save model
word2vec_model.save("custom_word2vec_hate.model")

print("Word2Vec model training complete!")


Word2Vec model training complete!


In [2]:
import numpy as np
import pandas as pd
import re
import nltk
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, Flatten
from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Download necessary NLTK resources
nltk.download('stopwords')
nltk.download('wordnet')

# Load dataset
train_df = pd.read_csv("train_hate.csv")

# Initialize stopwords and lemmatizer
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

# Function to preprocess text
def preprocess_text(text):
    text = text.lower()  # Convert to lowercase
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # Remove special characters & numbers
    words = text.split()  # Tokenize
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]  # Remove stopwords & lemmatize
    return " ".join(words)

# Apply preprocessing to dataset
train_df["Cleaned_Sentence"] = train_df["Sentence"].apply(preprocess_text)

# Load pre-trained Word2Vec model
word2vec_model = Word2Vec.load("custom_word2vec_hate.model")

# Tokenization and padding
tokenizer = Tokenizer()
tokenizer.fit_on_texts(train_df["Cleaned_Sentence"])  # Use cleaned text
sequences = tokenizer.texts_to_sequences(train_df["Cleaned_Sentence"])
max_len = max(len(seq) for seq in sequences)
X = pad_sequences(sequences, maxlen=max_len)

# Convert labels to numpy array
y = np.array(train_df["Tag"])

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create embedding matrix from Word2Vec
embedding_dim = 100
word_index = tokenizer.word_index
embedding_matrix = np.zeros((len(word_index) + 1, embedding_dim))
for word, i in word_index.items():
    if word in word2vec_model.wv:
        embedding_matrix[i] = word2vec_model.wv[word]

# Build FFNN model
model = Sequential([
    Embedding(input_dim=len(word_index) + 1, output_dim=embedding_dim, 
              weights=[embedding_matrix], trainable=False),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

# Compute class weights
class_weights = {0: 1, 1: len(y_train) / sum(y_train)}  # More weight for minority class

# Compile model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train model with class weighting
model.fit(X_train, y_train, class_weight=class_weights, epochs=10, batch_size=32, validation_data=(X_test, y_test))

# Save model using the recommended format
model.save("ffnn_text_classification_hate.keras")  # Updated format

print("FFNN model training complete and saved in .keras format!")


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\shrey\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\shrey\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Epoch 1/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.4410 - loss: 1.1638 - val_accuracy: 0.4385 - val_loss: 0.7836
Epoch 2/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.4480 - loss: 1.0701 - val_accuracy: 0.3825 - val_loss: 0.8426
Epoch 3/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.4351 - loss: 1.0580 - val_accuracy: 0.4549 - val_loss: 0.7386
Epoch 4/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.4725 - loss: 1.0423 - val_accuracy: 0.4413 - val_loss: 0.7780
Epoch 5/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.4830 - loss: 1.0139 - val_accuracy: 0.4426 - val_loss: 0.7370
Epoch 6/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.4764 - loss: 1.0108 - val_accuracy: 0.4385 - val_loss: 0.8005
Epoch 7/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.4856 - loss: 1.0026 - val_accuracy: 0.4481 - val_loss: 0.7916
Epoch 8/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.4900 - loss: 0.9900 - val_accuracy: 0.4604 - val_loss:

In [3]:
import numpy as np
import pandas as pd
import re
import string
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

# Function for text preprocessing (same as training data)
def preprocess_text(text):
    text = text.lower()  # Convert to lowercase
    text = re.sub(r'\d+', '', text)  # Remove numbers
    text = text.translate(str.maketrans('', '', string.punctuation))  # Remove punctuation
    text = re.sub(r'\s+', ' ', text).strip()  # Remove extra spaces
    return text

# Load validation dataset
val_df = pd.read_csv("val_hate.csv")

# Apply preprocessing (only if training was preprocessed)
val_df["Cleaned_Sentence"] = val_df["Sentence"].apply(preprocess_text)

# Tokenize and pad validation text
val_sequences = tokenizer.texts_to_sequences(val_df["Cleaned_Sentence"])
X_val = pad_sequences(val_sequences, maxlen=max_len)

# Get validation labels
y_val = np.array(val_df["Tag"])

# Predict on validation data
y_pred_probs = model.predict(X_val)
y_pred = (y_pred_probs > 0.5).astype(int)  # Convert probabilities to binary class (0 or 1)

# Compute Metrics
accuracy = accuracy_score(y_val, y_pred)
precision = precision_score(y_val, y_pred)
recall = recall_score(y_val, y_pred)
f1 = f1_score(y_val, y_pred)
conf_matrix = confusion_matrix(y_val, y_pred)

# Display Results
print("Model Evaluation on Validation Data:")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_val, y_pred))
print("\nConfusion Matrix:")
print(conf_matrix)


15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
Model Evaluation on Validation Data:
Accuracy: 0.4289
Precision: 0.3381
Recall: 0.7973
F1 Score: 0.4748

Classification Report:
              precision    recall  f1-score   support

           0       0.72      0.25      0.37       309
           1       0.34      0.80      0.47       148

    accuracy                           0.43       457
   macro avg       0.53      0.52      0.42       457
weighted avg       0.60      0.43      0.41       457


Confusion Matrix:
[[ 78 231]
 [ 30 118]]


# for sarcasm

In [4]:
import pandas as pd
from gensim.models import Word2Vec

# Load dataset
train_df = pd.read_csv("sarcasm_train.csv")

# Tokenize sentences (simple split method)
train_tokens = [sentence.lower().split() for sentence in train_df["Sentence"]]

# Train Word2Vec model
embedding_size = 100
word2vec_model = Word2Vec(sentences=train_tokens, vector_size=embedding_size, window=5, min_count=1, workers=4)

# Save model
word2vec_model.save("custom_word2vec_sarcasm.model")

print("Word2Vec model training complete!")


Word2Vec model training complete!


In [5]:
import numpy as np
import pandas as pd
import re
import nltk
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, Flatten
from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Download necessary NLTK resources
nltk.download('stopwords')
nltk.download('wordnet')

# Load dataset
train_df = pd.read_csv("sarcasm_train.csv")

# Initialize stopwords and lemmatizer
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

# Function to preprocess text
def preprocess_text(text):
    text = text.lower()  # Convert to lowercase
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # Remove special characters & numbers
    words = text.split()  # Tokenize
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]  # Remove stopwords & lemmatize
    return " ".join(words)

# Apply preprocessing to dataset
train_df["Cleaned_Sentence"] = train_df["Sentence"].apply(preprocess_text)

# Load pre-trained Word2Vec model
word2vec_model = Word2Vec.load("custom_word2vec_sarcasm.model")

# Tokenization and padding
tokenizer = Tokenizer()
tokenizer.fit_on_texts(train_df["Cleaned_Sentence"])  # Use cleaned text
sequences = tokenizer.texts_to_sequences(train_df["Cleaned_Sentence"])
max_len = max(len(seq) for seq in sequences)
X = pad_sequences(sequences, maxlen=max_len)

# Convert labels to numpy array
y = np.array(train_df["Tag"])

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create embedding matrix from Word2Vec
embedding_dim = 100
word_index = tokenizer.word_index
embedding_matrix = np.zeros((len(word_index) + 1, embedding_dim))
for word, i in word_index.items():
    if word in word2vec_model.wv:
        embedding_matrix[i] = word2vec_model.wv[word]

# Build FFNN model
model = Sequential([
    Embedding(input_dim=len(word_index) + 1, output_dim=embedding_dim, 
              weights=[embedding_matrix], trainable=False),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

# Compute class weights
class_weights = {0: 1, 1: len(y_train) / sum(y_train)}  # More weight for minority class

# Compile model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train model with class weighting
model.fit(X_train, y_train, class_weight=class_weights, epochs=10, batch_size=32, validation_data=(X_test, y_test))

# Save model using the recommended format
model.save("ffnn_text_classification_sarcasm.keras")  # Updated format

print("FFNN model training complete and saved in .keras format!")


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\shrey\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\shrey\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Epoch 1/10
105/105 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.5604 - loss: 1.3059 - val_accuracy: 0.7940 - val_loss: 0.3990
Epoch 2/10
105/105 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6594 - loss: 1.0331 - val_accuracy: 0.6286 - val_loss: 0.5951
Epoch 3/10
105/105 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6519 - loss: 1.0079 - val_accuracy: 0.7679 - val_loss: 0.4481
Epoch 4/10
105/105 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6779 - loss: 0.9360 - val_accuracy: 0.7464 - val_loss: 0.4551
Epoch 5/10
105/105 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.7194 - loss: 0.8401 - val_accuracy: 0.7881 - val_loss: 0.4117
Epoch 6/10
105/105 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.7427 - loss: 0.7866 - val_accuracy: 0.6952 - val_loss: 0.5575
Epoch 7/10
105/105 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.7285 - loss: 0.7863 - val_accuracy: 0.7869 - val_loss: 0.4216
Epoch 8/10
105/105 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.7523 - loss: 0.7672 - val_accuracy: 0.

In [6]:
import numpy as np
import pandas as pd
import re
import string
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

# Function for text preprocessing (same as training data)
def preprocess_text(text):
    text = text.lower()  # Convert to lowercase
    text = re.sub(r'\d+', '', text)  # Remove numbers
    text = text.translate(str.maketrans('', '', string.punctuation))  # Remove punctuation
    text = re.sub(r'\s+', ' ', text).strip()  # Remove extra spaces
    return text

# Load validation dataset
val_df = pd.read_csv("sarcasm_val.csv")

# Apply preprocessing (only if training was preprocessed)
val_df["Cleaned_Sentence"] = val_df["Sentence"].apply(preprocess_text)

# Tokenize and pad validation text
val_sequences = tokenizer.texts_to_sequences(val_df["Cleaned_Sentence"])
X_val = pad_sequences(val_sequences, maxlen=max_len)

# Get validation labels
y_val = np.array(val_df["Tag"])

# Predict on validation data
y_pred_probs = model.predict(X_val)
y_pred = (y_pred_probs > 0.5).astype(int)  # Convert probabilities to binary class (0 or 1)

# Compute Metrics
accuracy = accuracy_score(y_val, y_pred)
precision = precision_score(y_val, y_pred)
recall = recall_score(y_val, y_pred)
f1 = f1_score(y_val, y_pred)
conf_matrix = confusion_matrix(y_val, y_pred)

# Display Results
print("Model Evaluation on Validation Data:")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_val, y_pred))
print("\nConfusion Matrix:")
print(conf_matrix)


17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
Model Evaluation on Validation Data:
Accuracy: 0.8038
Precision: 0.2903
Recall: 0.7059
F1 Score: 0.4114

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.81      0.88       474
           1       0.29      0.71      0.41        51

    accuracy                           0.80       525
   macro avg       0.63      0.76      0.65       525
weighted avg       0.90      0.80      0.84       525


Confusion Matrix:
[[386  88]
 [ 15  36]]


# for humor

In [7]:
import pandas as pd
from gensim.models import Word2Vec

# Load dataset
train_df = pd.read_csv("humor_train.csv")

# Tokenize sentences (simple split method)
train_tokens = [sentence.lower().split() for sentence in train_df["Sentence"]]

# Train Word2Vec model
embedding_size = 100
word2vec_model = Word2Vec(sentences=train_tokens, vector_size=embedding_size, window=5, min_count=1, workers=4)

# Save model
word2vec_model.save("custom_word2vec_humor.model")

print("Word2Vec model training complete!")


Word2Vec model training complete!


In [8]:
import numpy as np
import pandas as pd
import re
import nltk
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, Flatten
from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Download necessary NLTK resources
nltk.download('stopwords')
nltk.download('wordnet')

# Load dataset
train_df = pd.read_csv("humor_train.csv")

# Initialize stopwords and lemmatizer
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

# Function to preprocess text
def preprocess_text(text):
    text = text.lower()  # Convert to lowercase
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # Remove special characters & numbers
    words = text.split()  # Tokenize
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]  # Remove stopwords & lemmatize
    return " ".join(words)

# Apply preprocessing to dataset
train_df["Cleaned_Sentence"] = train_df["Sentence"].apply(preprocess_text)

# Load pre-trained Word2Vec model
word2vec_model = Word2Vec.load("custom_word2vec_humor.model")

# Tokenization and padding
tokenizer = Tokenizer()
tokenizer.fit_on_texts(train_df["Cleaned_Sentence"])  # Use cleaned text
sequences = tokenizer.texts_to_sequences(train_df["Cleaned_Sentence"])
max_len = max(len(seq) for seq in sequences)
X = pad_sequences(sequences, maxlen=max_len)

# Convert labels to numpy array
y = np.array(train_df["Tag"])

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create embedding matrix from Word2Vec
embedding_dim = 100
word_index = tokenizer.word_index
embedding_matrix = np.zeros((len(word_index) + 1, embedding_dim))
for word, i in word_index.items():
    if word in word2vec_model.wv:
        embedding_matrix[i] = word2vec_model.wv[word]

# Build FFNN model
model = Sequential([
    Embedding(input_dim=len(word_index) + 1, output_dim=embedding_dim, 
              weights=[embedding_matrix], trainable=False),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

# Compute class weights
class_weights = {0: 1, 1: len(y_train) / sum(y_train)}  # More weight for minority class

# Compile model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train model with class weighting
model.fit(X_train, y_train, class_weight=class_weights, epochs=10, batch_size=32, validation_data=(X_test, y_test))

# Save model using the recommended format
model.save("ffnn_text_classification_humor.keras")  # Updated format

print("FFNN model training complete and saved in .keras format!")


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\shrey\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\shrey\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Epoch 1/10
59/59 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.5996 - loss: 0.8281 - val_accuracy: 0.6038 - val_loss: 0.6932
Epoch 2/10
59/59 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6188 - loss: 0.7693 - val_accuracy: 0.6229 - val_loss: 0.6825
Epoch 3/10
59/59 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6259 - loss: 0.7821 - val_accuracy: 0.6314 - val_loss: 0.6871
Epoch 4/10
59/59 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6354 - loss: 0.7802 - val_accuracy: 0.6123 - val_loss: 0.7175
Epoch 5/10
59/59 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6374 - loss: 0.7527 - val_accuracy: 0.6419 - val_loss: 0.6735
Epoch 6/10
59/59 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6290 - loss: 0.7555 - val_accuracy: 0.6271 - val_loss: 0.6611
Epoch 7/10
59/59 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6282 - loss: 0.7666 - val_accuracy: 0.6335 - val_loss: 0.6819
Epoch 8/10
59/59 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6403 - loss: 0.7505 - val_accuracy: 0.6335 - val_loss:

In [9]:
import numpy as np
import pandas as pd
import re
import string
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

# Function for text preprocessing (same as training data)
def preprocess_text(text):
    text = text.lower()  # Convert to lowercase
    text = re.sub(r'\d+', '', text)  # Remove numbers
    text = text.translate(str.maketrans('', '', string.punctuation))  # Remove punctuation
    text = re.sub(r'\s+', ' ', text).strip()  # Remove extra spaces
    return text

# Load validation dataset
val_df = pd.read_csv("humor_val.csv")

# Apply preprocessing (only if training was preprocessed)
val_df["Cleaned_Sentence"] = val_df["Sentence"].apply(preprocess_text)

# Tokenize and pad validation text
val_sequences = tokenizer.texts_to_sequences(val_df["Cleaned_Sentence"])
X_val = pad_sequences(val_sequences, maxlen=max_len)

# Get validation labels
y_val = np.array(val_df["Tag"])

# Predict on validation data
y_pred_probs = model.predict(X_val)
y_pred = (y_pred_probs > 0.5).astype(int)  # Convert probabilities to binary class (0 or 1)

# Compute Metrics
accuracy = accuracy_score(y_val, y_pred)
precision = precision_score(y_val, y_pred)
recall = recall_score(y_val, y_pred)
f1 = f1_score(y_val, y_pred)
conf_matrix = confusion_matrix(y_val, y_pred)

# Display Results
print("Model Evaluation on Validation Data:")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_val, y_pred))
print("\nConfusion Matrix:")
print(conf_matrix)


10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step
Model Evaluation on Validation Data:
Accuracy: 0.6407
Precision: 0.6296
Recall: 0.9659
F1 Score: 0.7623

Classification Report:
              precision    recall  f1-score   support

           0       0.76      0.16      0.26       119
           1       0.63      0.97      0.76       176

    accuracy                           0.64       295
   macro avg       0.69      0.56      0.51       295
weighted avg       0.68      0.64      0.56       295


Confusion Matrix:
[[ 19 100]
 [  6 170]]
